# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedant-Jagtap/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

march_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    data_files={"train": "fact_content_daily_performance/month=2026-03/data_0.parquet"},
    token=HF_TOKEN
)

march_df = march_ds.to_pandas()

print(march_df.shape)
march_df.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### Rule

Pages with low click-through performance and low organic traffic are candidates for content refresh.

### Action Label

REFRESH_CONTENT

### Reason Code

LOW_CTR_LOW_TRAFFIC

### Why

If a page receives impressions but relatively few clicks and low organic traffic, the content may be outdated, poorly optimized, or less relevant to current search intent.

In [3]:
march_df["ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
)

march_df["ctr_bucket"] = pd.cut(
    march_df["ctr"],
    bins=[-1, 0.01, 0.03, 0.05, 1],
    labels=["Very Low", "Low", "Medium", "High"]
)

ctr_table = (
    march_df
    .groupby("ctr_bucket")
    .agg(
        n=("ctr", "count"),
        avg_clicks=("gsc_clicks", "mean")
    )
)

print(ctr_table)

/tmp/ipykernel_1704/2863854166.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("ctr_bucket")


                  n  avg_clicks
ctr_bucket                     
Very Low    3414297    0.125529
Low          132328    2.231002
Medium        29724    1.743911
High          34712    1.330376


Verdict: CONFIRMED

Pages with very low CTR show weaker engagement and are reasonable candidates for refresh actions.

In [4]:
march_df["traffic_bucket"] = pd.cut(
    march_df["sessions_organic"].fillna(0),
    bins=[-1, 0, 10, 100, march_df["sessions_organic"].max()],
    labels=["Zero", "Low", "Medium", "High"]
)

traffic_table = (
    march_df
    .groupby("traffic_bucket", observed=False)
    .agg(
        n=("sessions_organic", "count"),
        avg_sessions=("sessions_organic", "mean")
    )
)

print(traffic_table)

                      n  avg_sessions
traffic_bucket                       
Zero            6609994      0.000000
Low              206946      2.298015
Medium             5659     18.460329
High                 38    141.473684


Verdict: CONFIRMED

The majority of pages have zero or very low organic traffic. Pages with low traffic are reasonable candidates for content refresh because they are attracting limited organic visibility.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import pandas as pd
import numpy as np
import os

# --------------------------------------------------
# Create manageable sample
# --------------------------------------------------

sample_df = march_df.sample(
    n=50000,
    random_state=42
).copy()

print("Sample Shape:", sample_df.shape)

# --------------------------------------------------
# Signal 1: CTR
# --------------------------------------------------

sample_df["ctr"] = (
    sample_df["gsc_clicks"] /
    sample_df["gsc_impressions"].replace(0, np.nan)
)

sample_df["ctr"] = sample_df["ctr"].fillna(0)

# --------------------------------------------------
# Score Component 1
# Low CTR -> Higher refresh score
# Weight = 70
# --------------------------------------------------

sample_df["ctr_score"] = (
    1 - sample_df["ctr"].clip(0, 1)
) * 70

# --------------------------------------------------
# Score Component 2
# Low organic traffic -> Higher refresh score
# Weight = 30
# --------------------------------------------------

max_sessions = sample_df["sessions_organic"].fillna(0).max()

sample_df["traffic_score"] = (
    1 -
    (
        sample_df["sessions_organic"].fillna(0)
        / max_sessions
    )
) * 30

# --------------------------------------------------
# Final Score
# --------------------------------------------------

sample_df["baseline_score"] = (
    sample_df["ctr_score"] +
    sample_df["traffic_score"]
)

# --------------------------------------------------
# Action + Reason Code
# --------------------------------------------------

sample_df["action_label"] = "REFRESH_CONTENT"

sample_df["reason_code"] = "LOW_CTR_LOW_TRAFFIC"

# --------------------------------------------------
# Ranked Queue
# --------------------------------------------------

queue = sample_df.sort_values(
    by="baseline_score",
    ascending=False
)

# --------------------------------------------------
# Save CSV
# --------------------------------------------------

os.makedirs(
    "work/outputs",
    exist_ok=True
)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ CSV written successfully")
print("Location: work/outputs/baseline_action_score.csv")

# --------------------------------------------------
# Show Top 10
# --------------------------------------------------

top10 = queue[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "sessions_organic",
        "baseline_score",
        "action_label",
        "reason_code"
    ]
].head(10)

print("\nTOP 10 PAGES FOR REFRESH\n")
print(top10)

top10

Sample Shape: (50000, 33)
✅ CSV written successfully
Location: work/outputs/baseline_action_score.csv

TOP 10 PAGES FOR REFRESH

                  content_hash_id  gsc_impressions  gsc_clicks  \
2341128  content_df9c7821a36abad2                0           0   
1533444  content_78a7a76a73d1134e                1           0   
5059310  content_f35f709a99068625                0           0   
923508   content_31853e2d0bcc8015                0           0   
6513819  content_7485b9fd266251f2                0           0   
8845843  content_ad60ea58436ef2e3              107           0   
1443577  content_2f9cc3c6ebed9a6b                0           0   
65532    content_98f9b9c8326bbb4f               19           0   
5094035  content_d8730c372c1f4b24                4           0   
9446016  content_1ad28ad024060d6a                0           0   

         sessions_organic  baseline_score     action_label  \
2341128               0.0           100.0  REFRESH_CONTENT   
1533444             

,content_hash_id,gsc_impressions,gsc_clicks,sessions_organic,baseline_score,action_label,reason_code
2341128,content_df9c7821a36abad2,0,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
1533444,content_78a7a76a73d1134e,1,0,NaN,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
5059310,content_f35f709a99068625,0,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
923508,content_31853e2d0bcc8015,0,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
6513819,content_7485b9fd266251f2,0,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
8845843,content_ad60ea58436ef2e3,107,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
1443577,content_2f9cc3c6ebed9a6b,0,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
65532,content_98f9b9c8326bbb4f,19,0,NaN,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
5094035,content_d8730c372c1f4b24,4,0,0.0,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC
9446016,content_1ad28ad024060d6a,0,0,NaN,100.0,REFRESH_CONTENT,LOW_CTR_LOW_TRAFFIC


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-10 Review

1. Action: REFRESH_CONTENT
   Reason: No clicks and no organic traffic.
   What would make it wrong: The page may be new and not yet indexed.

2. Action: REFRESH_CONTENT
   Reason: Extremely low CTR and no traffic.
   What would make it wrong: The page may target a very niche keyword.

3. Action: REFRESH_CONTENT
   Reason: No engagement signals detected.
   What would make it wrong: Tracking data may be incomplete.

4. Action: REFRESH_CONTENT
   Reason: No clicks despite available impressions.
   What would make it wrong: Seasonal search demand may be affecting traffic.

5. Action: REFRESH_CONTENT
   Reason: Low visibility and low engagement.
   What would make it wrong: The page may not be intended for organic traffic.

6. Action: REFRESH_CONTENT
   Reason: Weak CTR and weak traffic signals.
   What would make it wrong: Search intent may have recently changed.

7. Action: REFRESH_CONTENT
   Reason: No measurable engagement.
   What would make it wrong: Analytics data could be delayed.

8. Action: REFRESH_CONTENT
   Reason: Low traffic and no clicks.
   What would make it wrong: The page could support another high-performing page.

9. Action: REFRESH_CONTENT
   Reason: Poor search performance indicators.
   What would make it wrong: The page may already be scheduled for an update.

10. Action: REFRESH_CONTENT
    Reason: Combined low CTR and low organic sessions.
    What would make it wrong: The page may serve a non-search business purpose.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Several top-ranked pages have zero impressions and zero clicks. These pages may not actually require a content refresh. Some may be newly created, not indexed, or have incomplete tracking data. This is a weakness of the baseline rule.

## Leakage Check

No future performance information was used.

No product flags were used.

The rule only used:

- gsc_impressions
- gsc_clicks
- sessions_organic

These signals are available before making the refresh decision.

The baseline is intended as a decision-support tool and not a final prediction model.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/

In [6]:
print(march_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'ctr', 'ctr_bucket', 'traffic_bucket']


In [11]:
queue["baseline_score"].describe()



,baseline_score
count,50000.000000
mean,99.894929
std,1.438638
min,29.104478
25%,100.000000
50%,100.000000
75%,100.000000
max,100.000000


In [12]:
queue["baseline_score"].quantile([0.8, 0.9, 0.95])

,baseline_score
0.80,100.0
0.90,100.0
0.95,100.0
